In [1]:
print("Python is working!")

Python is working!


In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../Data/Raw/Commodity_dataset_raw.csv")

In [4]:
df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_x0020_Price,Max_x0020_Price,Modal_x0020_Price
0,Keralam,Kozhikode(Calicut),Mukkom Market,Banana,Other,Large,28/07/2026,5200.0,5600.0,5500.0
1,Keralam,Kozhikode(Calicut),Mukkom Market,Onion,Onion,FAQ,28/07/2026,2800.0,3200.0,3000.0
2,Keralam,Kozhikode(Calicut),Mukkom Market,Cucumbar(Kheera),Cucumbar,FAQ,28/07/2026,1800.0,2100.0,1900.0
3,Keralam,Kozhikode(Calicut),Mukkom Market,Snakeguard,Snakeguard,FAQ,28/07/2026,3000.0,3300.0,3100.0
4,Haryana,Sonipat,Ganaur APMC,Tomato,Tomato,FAQ,28/07/2026,2000.0,3000.0,2500.0


**Checking dataset size** 
: before ETL just check what is inside the data 

In [5]:
df.shape

(7181, 10)

In [6]:
df.columns #seeing the column names

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
       'Arrival_Date', 'Min_x0020_Price', 'Max_x0020_Price',
       'Modal_x0020_Price'],
      dtype='object')

In [7]:
df.info() #checking data types 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7181 entries, 0 to 7180
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   State              7181 non-null   object 
 1   District           7181 non-null   object 
 2   Market             7181 non-null   object 
 3   Commodity          7181 non-null   object 
 4   Variety            7181 non-null   object 
 5   Grade              7181 non-null   object 
 6   Arrival_Date       7181 non-null   object 
 7   Min_x0020_Price    7181 non-null   float64
 8   Max_x0020_Price    7181 non-null   float64
 9   Modal_x0020_Price  7181 non-null   float64
dtypes: float64(3), object(7)
memory usage: 561.1+ KB


In [8]:
# checking null values
df.isnull().sum()

State                0
District             0
Market               0
Commodity            0
Variety              0
Grade                0
Arrival_Date         0
Min_x0020_Price      0
Max_x0020_Price      0
Modal_x0020_Price    0
dtype: int64

In [9]:
#Checking duplicate rows
df.duplicated().sum()

np.int64(0)

In [10]:
#statistics of the data(min max values,etc.)
df.describe()

,Min_x0020_Price,Max_x0020_Price,Modal_x0020_Price
count,7181.000000,7181.000000,7181.000000
mean,5182.060971,5854.736689,5519.210416
std,4856.456855,5337.476386,5076.441092
min,1.000000,1.000000,1.000000
25%,3000.000000,3000.000000,3000.000000
50%,4000.000000,4500.000000,4300.000000
75%,6000.000000,6400.000000,6000.000000
max,100000.000000,105000.000000,102500.000000


**START OF ETL**

In [11]:
df_clean = df.copy() # creating dataset copy

In [12]:
#setting clean column names
df_clean.columns = (
    df_clean.columns
    .str.replace("_x0020_", "_", regex=False)
    .str.strip()
    .str.lower()
)

In [13]:
df_clean.columns

Index(['state', 'district', 'market', 'commodity', 'variety', 'grade',
       'arrival_date', 'min_price', 'max_price', 'modal_price'],
      dtype='object')

In [14]:
#Remove unnecessary spaces from text : removing white space before and after words
text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

In [15]:
#Convert date column
df_clean.columns

Index(['state', 'district', 'market', 'commodity', 'variety', 'grade',
       'arrival_date', 'min_price', 'max_price', 'modal_price'],
      dtype='object')

In [16]:
df_clean["arrival_date"] = pd.to_datetime(
    df_clean["arrival_date"],
    errors="coerce"
)

C:\Users\Devyani\AppData\Local\Temp\ipykernel_8556\3165194439.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_clean["arrival_date"] = pd.to_datetime(


In [17]:
df_clean["arrival_date"].isnull().sum()

np.int64(0)

Make price columns numeric

In [18]:
price_columns = ["min_price", "max_price", "modal_price"]

for col in price_columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

In [19]:
df_clean[price_columns].dtypes

min_price      float64
max_price      float64
modal_price    float64
dtype: object

In [20]:
df_clean.isnull().sum()

state           0
district        0
market          0
commodity       0
variety         0
grade           0
arrival_date    0
min_price       0
max_price       0
modal_price     0
dtype: int64

In [21]:
df_clean.dtypes

state                   object
district                object
market                  object
commodity               object
variety                 object
grade                   object
arrival_date    datetime64[ns]
min_price              float64
max_price              float64
modal_price            float64
dtype: object

there are no missing values in any column.we do NOT need imputation

**Data Validation**

In [22]:
#Check duplicates
df_clean.duplicated().sum()

np.int64(0)

In [23]:
#Check price logic:Min Price ≤ Modal Price ≤ Max Price
(df_clean["min_price"] > df_clean["modal_price"]).sum()


np.int64(0)

In [24]:
(df_clean["modal_price"] > df_clean["max_price"]).sum()

np.int64(0)

In [25]:
#Check negative prices
(df_clean[["min_price", "max_price", "modal_price"]] < 0).sum()

min_price      0
max_price      0
modal_price    0
dtype: int64

In [26]:
#Check actual date range of dataset
df_clean["arrival_date"].min(), df_clean["arrival_date"].max()

(Timestamp('2026-07-28 00:00:00'), Timestamp('2026-07-28 00:00:00'))

**Transformation**

In [27]:
df_clean.duplicated().sum()

np.int64(0)

**Create useful business metric**

In [28]:
#useful derived metric is price spread:
df_clean["price_spread"] = (
    df_clean["max_price"] - df_clean["min_price"]
)

In [29]:
#Create price range percentage
df_clean["price_spread_pct"] = (
    df_clean["price_spread"] / df_clean["modal_price"]
) * 100 #This lets us identify commodities/markets where prices vary widely.

In [30]:
#Extract date components
df_clean["year"] = df_clean["arrival_date"].dt.year
df_clean["month"] = df_clean["arrival_date"].dt.month
df_clean["day"] = df_clean["arrival_date"].dt.day

In [31]:
#Check the transformed data
df_clean.head()

,state,district,market,commodity,variety,grade,arrival_date,min_price,max_price,modal_price,price_spread,price_spread_pct,year,month,day
0,Keralam,Kozhikode(Calicut),Mukkom Market,Banana,Other,Large,2026-07-28,5200.0,5600.0,5500.0,400.0,7.272727,2026,7,28
1,Keralam,Kozhikode(Calicut),Mukkom Market,Onion,Onion,FAQ,2026-07-28,2800.0,3200.0,3000.0,400.0,13.333333,2026,7,28
2,Keralam,Kozhikode(Calicut),Mukkom Market,Cucumbar(Kheera),Cucumbar,FAQ,2026-07-28,1800.0,2100.0,1900.0,300.0,15.789474,2026,7,28
3,Keralam,Kozhikode(Calicut),Mukkom Market,Snakeguard,Snakeguard,FAQ,2026-07-28,3000.0,3300.0,3100.0,300.0,9.677419,2026,7,28
4,Haryana,Sonipat,Ganaur APMC,Tomato,Tomato,FAQ,2026-07-28,2000.0,3000.0,2500.0,1000.0,40.000000,2026,7,28


In [32]:
df_clean.shape

(7181, 15)

In [33]:
df_clean[[
    "min_price",
    "max_price",
    "modal_price",
    "price_spread",
    "price_spread_pct"
]].describe()

,min_price,max_price,modal_price,price_spread,price_spread_pct
count,7181.000000,7181.000000,7181.000000,7181.000000,7181.000000
mean,5182.060971,5854.736689,5519.210416,672.675718,13.518085
std,4856.456855,5337.476386,5076.441092,1119.874934,19.185876
min,1.000000,1.000000,1.000000,0.000000,0.000000
25%,3000.000000,3000.000000,3000.000000,200.000000,3.703704
50%,4000.000000,4500.000000,4300.000000,500.000000,10.714286
75%,6000.000000,6400.000000,6000.000000,800.000000,18.181818
max,100000.000000,105000.000000,102500.000000,21000.000000,968.750000


**Final ETL validation**

In [34]:
df_clean.isnull().sum()

state               0
district            0
market              0
commodity           0
variety             0
grade               0
arrival_date        0
min_price           0
max_price           0
modal_price         0
price_spread        0
price_spread_pct    0
year                0
month               0
day                 0
dtype: int64

In [35]:
df_clean.duplicated().sum()

np.int64(0)

In [37]:
#Save the processed dataset :create the actual ETL output
df_clean.to_csv(
    "../Data/Processed/commodity_cleaned.csv",
    index=False
)


PermissionError: [Errno 13] Permission denied: '../Data/Processed/commodity_cleaned.csv'

In [38]:
import pandas as pd
import mysql.connector

In [39]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Devyani@999",
    database="commodity_price_dw"
)

In [40]:
tables = [
    "dim_date",
    "dim_location",
    "dim_commodity",
    "dim_variety",
    "dim_grade",
    "fact_market_price"
]

with pd.ExcelWriter("commodity_price_dw.xlsx", engine="openpyxl") as writer:
    for table in tables:
        df = pd.read_sql(f"SELECT * FROM {table}", conn)
        df.to_excel(writer, sheet_name=table[:31], index=False)

conn.close()

C:\Users\Devyani\AppData\Local\Temp\ipykernel_8556\3631835418.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table}", conn)
